In [1]:
# First install mygene: 
!pip install mygene
import pandas as pd
import mygene

def convert_genes_with_mygene(input_file, output_file):
    """
    Convert mouse Ensembl IDs to gene symbols using mygene library
    This is much more reliable than BioMart API calls!
    """
    print("=== Gene Conversion using mygene ===")
    
    # Read the CSV file
    print(f"Reading file: {input_file}")
    df = pd.read_csv(input_file)
    print(f"✓ Loaded {len(df):,} genes")
    print(f"✓ Columns: {list(df.columns)}")
    
    # Check for Genes column
    if 'Genes' not in df.columns:
        print("✗ 'Genes' column not found!")
        print("Available columns:", df.columns.tolist())
        return None
    
    # Get unique gene IDs
    gene_ids = df['Genes'].dropna().astype(str).unique().tolist()
    print(f"✓ Found {len(gene_ids):,} unique gene IDs")
    print(f"Sample IDs: {gene_ids[:5]}")
    
    # Initialize MyGeneInfo
    print("Initializing mygene...")
    mg = mygene.MyGeneInfo()
    
    # Convert genes in batches (mygene handles this efficiently)
    print("Converting Ensembl IDs to gene symbols...")
    print("This may take a few minutes for large datasets...")
    
    # Query mygene for gene symbols
    # Use querymany for multiple genes at once
    results = mg.querymany(
        gene_ids,
        scopes='ensembl.gene',  # Input: Ensembl gene IDs
        fields='symbol',        # Output: Gene symbols
        species='mouse',        # Species: mouse
        returnall=True         # Return both found and not found
    )
    
    print(f"✓ Query completed!")
    print(f"✓ Found: {len(results['out'])}")
    print(f"✓ Not found: {len(results['missing'])}")
    print(f"✓ Duplicates: {len(results['dup'])}")
    
    # Process results
    conversion_dict = {}
    for result in results['out']:
        if 'symbol' in result and 'query' in result:
            conversion_dict[result['query']] = result['symbol']
    
    print(f"✓ Successfully mapped {len(conversion_dict):,} genes")
    
    # Show some examples
    print("\nSample conversions:")
    for i, (ensembl_id, symbol) in enumerate(list(conversion_dict.items())[:10]):
        print(f"  {ensembl_id} → {symbol}")
    
    # Add gene symbols to dataframe
    df['Gene_Symbol'] = df['Genes'].map(conversion_dict)
    
    # Reorder columns to put gene symbol near the front
    cols = df.columns.tolist()
    cols.remove('Gene_Symbol')
    cols.insert(1, 'Gene_Symbol')  # Insert after 'Genes' column
    df_final = df[cols]
    
    # Save results
    df_final.to_csv(output_file, index=False)
    
    # Show statistics
    converted = df_final['Gene_Symbol'].notna().sum()
    total = len(df_final)
    
    print(f"\n=== Conversion Summary ===")
    print(f"Total genes: {total:,}")
    print(f"Successfully converted: {converted:,}")
    print(f"Not converted: {total - converted:,}")
    print(f"Success rate: {converted/total*100:.1f}%")
    print(f"Output saved to: {output_file}")
    
    # Show final examples
    print(f"\nFirst 10 results:")
    examples = df_final[['Genes', 'Gene_Symbol']].head(10)
    print(examples.to_string(index=False))
    
    # Show genes that didn't convert
    if total - converted > 0:
        no_symbols = df_final[df_final['Gene_Symbol'].isna()]['Genes'].head(5).tolist()
        print(f"\nFirst 5 genes without symbols: {no_symbols}")
    
    return df_final

def quick_test_mygene():
    """Test mygene with a few known genes"""
    print("=== Quick Test with mygene ===")
    
    # Test with known mouse genes
    test_genes = [
        'ENSMUSG00000000001',  # Gnai3
        'ENSMUSG00000000028',  # Cdc45
        'ENSMUSG00000000056',  # Narf
        'ENSMUSG00000000078',  # Klf6
        'ENSMUSG00000000088'   # Cox5a
    ]
    
    print(f"Testing with {len(test_genes)} genes:")
    for gene in test_genes:
        print(f"  {gene}")
    
    try:
        mg = mygene.MyGeneInfo()
        results = mg.querymany(
            test_genes,
            scopes='ensembl.gene',
            fields='symbol',
            species='mouse'
        )
        
        print(f"\n✓ Test successful! Results:")
        for result in results:
            if 'symbol' in result:
                print(f"  {result['query']} → {result['symbol']}")
            else:
                print(f"  {result['query']} → (no symbol found)")
                
        return True
        
    except Exception as e:
        print(f"✗ Test failed: {e}")
        return False

def convert_with_getgenes(gene_ids):
    """
    Alternative method using getgenes() for smaller lists
    """
    print("=== Using getgenes() method ===")
    
    mg = mygene.MyGeneInfo()
    
    # Use getgenes for detailed information
    results = mg.getgenes(
        gene_ids,
        fields='symbol,name,ensembl.gene',
        species='mouse'
    )
    
    print("Detailed results:")
    for result in results:
        if result:  # Some results might be None
            ensembl_id = result.get('ensembl', {}).get('gene', 'N/A')
            symbol = result.get('symbol', 'N/A')
            name = result.get('name', 'N/A')
            print(f"  {ensembl_id} → {symbol} ({name})")

# Main execution
if __name__ == "__main__":
    # First, install mygene if not already installed
    try:
        import mygene
        print("✓ mygene library is available")
    except ImportError:
        print("✗ mygene not found. Please install it:")
        print("pip install mygene")
        exit(1)
    
    # Quick test first
    print("Running quick test...")
    if quick_test_mygene():
        print("\n✓ Quick test passed! Proceeding with full conversion...")
        
        # Full conversion
        input_file = "/Users/adityaelayavalli/Downloads/normalized_counts.csv"
        output_file = "/Users/adityaelayavalli/Downloads/normalized_counts_with_symbols_mygene.csv"
        
        result = convert_genes_with_mygene(input_file, output_file)
        
        if result is not None:
            print("\n🎉 Conversion completed successfully!")
        else:
            print("\n❌ Conversion failed. Check your input file.")
            
    else:
        print("\n❌ Quick test failed. Check your internet connection and try again.")

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


✓ mygene library is available
Running quick test...
=== Quick Test with mygene ===
Testing with 5 genes:
  ENSMUSG00000000001
  ENSMUSG00000000028
  ENSMUSG00000000056
  ENSMUSG00000000078
  ENSMUSG00000000088


Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed



✓ Test successful! Results:
  ENSMUSG00000000001 → Gnai3
  ENSMUSG00000000028 → Cdc45
  ENSMUSG00000000056 → Narf
  ENSMUSG00000000078 → Klf6
  ENSMUSG00000000088 → Cox5a

✓ Quick test passed! Proceeding with full conversion...
=== Gene Conversion using mygene ===
Reading file: /Users/adityaelayavalli/Downloads/normalized_counts.csv
✓ Loaded 29,993 genes
✓ Columns: ['Genes', 'X527_16_B_RNA_GT25.02583', 'X527_16_Bl_RNA_GT25.02549', 'X527_16_Lg_RNA_GT25.02595', 'X527_16_SI_RNA_GT25.02555', 'X527_18_B_RNA_GT25.02582', 'X527_18_Lg_RNA_GT25.02576', 'X527_18_SI_RNA_GT25.02574', 'X527_2_B_RNA_GT25.02551', 'X527_2_Bl_RNA_GT25.02553', 'X527_2_Lg_RNA_GT25.02557', 'X527_2_SI_RNA_GT25.02588', 'X527_20_B_RNA_GT25.02552', 'X527_20_Bl_RNA_GT25.02560', 'X527_20_Lg_RNA_GT25.02559', 'X527_20_SI_RNA_GT25.02593', 'X527_21_B_RNA_GT25.02567', 'X527_21_Bl_RNA_GT25.02575', 'X527_21_Lg_RNA_GT25.02563', 'X527_21_SI_RNA_GT25.02611', 'X527_25_B_RNA_GT25.02546', 'X527_25_Bl_RNA_GT25.02597', 'X527_25_Lg_RNA_GT25.0

1 input query terms found dup hits:	[('ENSMUSG00000072694', 2)]
452 input query terms found no hit:	['ENSMUSG00000003178', 'ENSMUSG00000004613', 'ENSMUSG00000011052', 'ENSMUSG00000022591', 'ENSMUSG000


✓ Query completed!
✓ Found: 29994
✓ Not found: 452
✓ Duplicates: 1
✓ Successfully mapped 29,523 genes

Sample conversions:
  ENSMUSG00000000001 → Gnai3
  ENSMUSG00000000003 → Pbsn
  ENSMUSG00000000028 → Cdc45
  ENSMUSG00000000037 → Scml2
  ENSMUSG00000000049 → Apoh
  ENSMUSG00000000056 → Narf
  ENSMUSG00000000058 → Cav2
  ENSMUSG00000000078 → Klf6
  ENSMUSG00000000085 → Scmh1
  ENSMUSG00000000088 → Cox5a

=== Conversion Summary ===
Total genes: 29,993
Successfully converted: 29,523
Not converted: 470
Success rate: 98.4%
Output saved to: /Users/adityaelayavalli/Downloads/normalized_counts_with_symbols_mygene.csv

First 10 results:
             Genes Gene_Symbol
ENSMUSG00000000001       Gnai3
ENSMUSG00000000003        Pbsn
ENSMUSG00000000028       Cdc45
ENSMUSG00000000037       Scml2
ENSMUSG00000000049        Apoh
ENSMUSG00000000056        Narf
ENSMUSG00000000058        Cav2
ENSMUSG00000000078        Klf6
ENSMUSG00000000085       Scmh1
ENSMUSG00000000088       Cox5a

First 5 genes withou